# Science Paper Analyzer - tutorial


This file explains every file in the project in simple words.


## Project structure


- `science_paper_analyzer.ipynb` - main Colab notebook (3 clicks)


- `analyzer.py` - the brain of the program


- `parsers/` - 7 files for 10 websites


- `analyzers/` - 3 checkers (citation, text, journal)


- `exporters/` - save to CSV and Excel


- `app.py` - web interface (Streamlit)


- `requirements.txt` - libraries needed


---


## 1. science_paper_analyzer.ipynb - main Colab


Three steps. Press Run three times.


### Step 1: Download code and install libraries


This cell downloads the code from GitHub and installs required Python packages.


In [ ]:
import subprocess, sys, os, importlib

REPO_URL = "https://github.com/DmitPerson42/science-paper-analyzer.git"
REPO_DIR = "science-paper-analyzer"

if os.path.exists(REPO_DIR):
    import shutil
    shutil.rmtree(REPO_DIR, ignore_errors=True)

subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True,
               capture_output=True, text=True)
os.chdir(REPO_DIR)

required = ["requests", "pandas", "openpyxl", "lxml", "beautifulsoup4"]
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing,
                   check=True, capture_output=True, text=True)

print("Step 1 done!")

**Shutil.rmtree** removes old code (to always have latest version).


**Git clone** downloads code from GitHub.


**Importlib.find_spec** checks if a package is installed.


**Pip install** installs only missing packages.


### Step 2: Search and analyze


This cell searches for articles and analyzes them.


In [ ]:
from analyzer import PaperAnalyzer

QUERY = "filtering generative text data language model collapse"
MAX_PER_SOURCE = 5

print(f"Searching: {QUERY}")
analyzer = PaperAnalyzer(verbose=True)
papers = analyzer.collect_papers(QUERY, max_per_source=MAX_PER_SOURCE)

if not papers:
    print("No articles found. Try different keywords.")
    import pandas as pd
    df = pd.DataFrame(columns=["title","authors","year","source",
                               "url","abstract","citation_score",
                               "text_score","journal_score",
                               "overall_score","verdict"])
else:
    df = analyzer.analyze_papers(papers)
    print(f"Analyzed {len(df)} articles.")
    
    display_cols = ["title", "authors", "year", "source",
                    "overall_score", "verdict"]
    df_view = df[display_cols].copy()
    df_view.columns = ["Title", "Authors", "Year", "Source", "Score", "Verdict"]
    display(df_view)
    
    total = len(df)
    rc = len(df[df["verdict"] == "Real"])
    sc = len(df[df["verdict"] == "Suspicious"])
    fc = len(df[df["verdict"] == "Fake"])
    print(f"Total: {total}  Real: {rc}  Suspicious: {sc}  Fake: {fc}")
    
print("Step 2 done!")

**PaperAnalyzer(verbose=True)** creates the main object.


**collect_papers()** goes through all 10 websites and collects articles.


**analyze_papers()** runs 3 checks on each article.


**display()** shows a nice table in Colab.


### Step 3: Download results


Saves to CSV and Excel, then downloads to your computer.


In [ ]:
if "df" in dir() and len(df) > 0:
    csv_path = "analysis_results.csv"
    xlsx_path = "analysis_results.xlsx"
    analyzer.export_csv(df, csv_path)
    analyzer.export_excel(df, xlsx_path)
    from google.colab import files
    files.download(csv_path)
    print("CSV downloaded!")
else:
    print("No data. Run Step 2 first.")

**dir()** checks if the df variable exists (prevents errors if Step 2 was skipped).


**files.download()** triggers the browser to download the file.


---


## 2. analyzer.py - the brain


This file connects everything together.


### Streamlit fallback (important!)


The file works in two modes: web (with Streamlit) and Colab (without Streamlit).


If Streamlit is not installed, a dummy class is used:


In [ ]:
try:
    import streamlit as st
    _has_streamlit = True
except ImportError:
    class _Stub:
        def progress(self, *a, **kw): return self
        def empty(self, *a, **kw): return self
        def warning(self, msg): print(f"[WARNING] {msg}")
        def __getattr__(self, name): return lambda *a, **kw: None
    st = _Stub()
    _has_streamlit = False

**Why:** Without this, `st.progress(0)` in Colab would crash with ModuleNotFoundError.


The stub class has all the same methods but they do nothing.


### PaperAnalyzer class


In [ ]:
class PaperAnalyzer:
    def __init__(self, verbose=True):
        self.parsers = self._init_parsers()      # 10 parsers
        self.analyzers = self._init_analyzers()  # 3 analyzers
        self.exporter_csv = CSVExporter()
        self.exporter_xlsx = ExcelExporter()
        self.verbose = verbose

**10 parsers** = one for each scientific website.


**3 analyzers** = citation checker, text checker, journal checker.


**verbose** = if True, prints progress to console.


### collect_papers() - search all websites


In [ ]:
def collect_papers(self, query, max_per_source=5):
    all_papers = []
    for source_key, parser in self.parsers.items():
        try:
            papers = parser.search(query, max_results=max_per_source)
            for p in papers:
                p["source"] = source_key
            all_papers.extend(papers)
            time.sleep(0.3)
        except Exception as e:
            print(f"[WARNING] {source_key}: {e}")

    # Remove duplicates (same title)
    seen = set()
    unique = []
    for p in all_papers:
        key = p.get("title", "").lower().strip()[:100]
        if key and key not in seen:
            seen.add(key)
            unique.append(p)
    return unique

**What happens:**


1. Loop through all 10 parsers


2. Each parser searches the query on its website


3. Each article gets a `source` tag (where it came from)


4. Wait 0.3 seconds between requests (to not overload servers)


5. Remove duplicates (same title from different sources)


### analyze_papers() - check each article


In [ ]:
def analyze_papers(self, papers):
    results = []
    for paper in papers:
        cs, cd = self.analyzers["citation"].analyze(paper)
        ts, td = self.analyzers["text"].analyze(paper)
        js, jd = self.analyzers["journal"].analyze(paper)

        avg = (cs + ts + js) / 3

        if avg >= 0.7:    v = "Real"
        elif avg >= 0.4:  v = "Suspicious"
        else:             v = "Fake"

        results.append({
            "title": paper.get("title"),
            "overall_score": round(avg, 2),
            "verdict": v,
        })
    return pd.DataFrame(results)

**How verdict works:**


- All 3 scores are averaged: (citation + text + journal) / 3


- If average >= 0.70 -> Real (trustworthy)


- If average 0.40-0.70 -> Suspicious (check manually)


- If average < 0.40 -> Fake (likely fabricated)


---


## 3. parsers/ - downloading articles


Each parser is a class with a `search(query)` method that returns a list of dictionaries.


### 3.1 arxiv_parser.py


arXiv has a free XML API. Endpoint: `http://export.arxiv.org/api/query`


**Key features:**


- Retry logic: if server says 429 (too many requests), wait 3-6-9 seconds


- arXiv returns XML, not JSON. Uses ElementTree to parse


- `extract_year()` converts date like "2023-01-15" to "2023"


- Limits authors to 5 to keep table clean


### 3.2 semantic_scholar.py


**This is the most important parser.** It is the only free source of citation counts.


Key field: `citationCount` - tells how many times an article was cited.


Without this, the CitationAnalyzer cannot work.


### 3.3 openreview_parser.py - **BUG WAS HERE**


OpenReview API returns timestamps in **milliseconds** (e.g., 1652000000000).


**Old code:** `str(1652000000000)[:4]` -> "1652" (WRONG - showed year 1652)


**Fixed code:** `extract_year(1652000000000)` -> "2022" (CORRECT)


The `extract_year()` function detects it's a millisecond timestamp and converts it.


### 3.4 aclanthology.py


Two modes:


1. Try the REST API first (fast, structured data)


2. If API fails, scrape the HTML website (slow, but works)


### 3.5 jmlr_parser.py


JMLR (Journal of Machine Learning Research) has no API - only HTML.


Years are determined from volume numbers: v1=2000, v2=2001, v22=2021


Formula: `year = 2000 + volume - 1`


### 3.6 crossref_fallback.py


Used for 4 websites that have no API:


- ResearchGate


- eLibrary.ru


- Dissercat.com


- FIPS.ru


All search through CrossRef - a central database of scientific publications.


Each gets a `source_context` tag so you know which source it came from.


### 3.7 cyberleninka.py


Russian scientific library. No API, scrapes HTML.


Needs a browser-like User-Agent to avoid being blocked.


### 3.8 utils.py - extract_year() function


This is the most important utility. It handles all date formats:


In [ ]:
def extract_year(value):
    if value is None or value == "":
        return ""
    if isinstance(value, (int, float)):
        if 1900 <= value <= 2050:
            return str(int(value))      # 2023 -> "2023"
        if value > 10**9:
            if value > 10**12:
                value = value / 1000    # ms to seconds
            from datetime import datetime
            return str(datetime.fromtimestamp(value).year)
    import re
    m = re.search(r'\b(19\d\d|20\d\d)\b', str(value))
    return m.group(1) if m else "" 

**Input -> Output examples:**


- 2023 -> "2023" (plain number)


- "2023-01-15" -> "2023" (ISO date string)


- "2023 year" -> "2023" (text with year)


- 1672531200 -> "2023" (Unix timestamp in seconds)


- 1652000000000 -> "2022" (Unix timestamp in milliseconds)


- None -> "" (empty)


---


## 4. analyzers/ - checking for fakes


Three analyzers, each returns (score, explanation).


### 4.1 citation_analyzer.py


Checks how many times an article was cited.


**Logic:**


- Old article (3+ years) with 0 citations -> suspicious (score 0.2)


- New article with 0 citations -> neutral (score 0.5)


- 1-5 citations -> acceptable (score 0.7)


- 5-15 citations -> good (score 0.85)


- 15+ citations -> excellent (score 0.95)


### 4.2 text_analyzer.py


Detects AI-generated text without using neural networks.


Uses 5 heuristics:


1. **AI markers** - phrases like "As an AI", "In conclusion", "State-of-the-art"


2. **TTR (Type-Token Ratio)** - humans use more diverse vocabulary (0.65) than AI (0.4)


3. **Repeated n-grams** - AI repeats same 3-word patterns


4. **Sentence length uniformity** - AI writes same-length sentences, humans vary


5. **Stop word ratio** - AI uses more filler words (the, a, of, and)


### 4.3 journal_analyzer.py


Checks against **Beall's List** - ~180 known predatory journals.


Also checks trusted venues (Nature, Science, IEEE, arXiv, etc.).


Trusted venue -> score 0.95. Predatory -> score 0.15. Unknown -> 0.60.


---


## 5. exporters/ - saving results


### csv_exporter.py


Saves to CSV with `utf-8-sig` encoding (so Excel shows Russian letters correctly).


### excel_exporter.py


Saves to Excel with colors: green=Real, yellow=Suspicious, red=Fake.


Also includes explanation columns (why each score was given).


---


## 6. app.py - web interface (Streamlit)


Run with: `streamlit run app.py`


Features:


- Sidebar with settings (topic, number of sources, checkboxes)


- Results table with green/yellow/red colors


- Detail view: click an article to see 3 tabs (Overview, Analysis, Text)


- Export buttons: CSV and Excel


---


## How it all connects


```


You type a topic


      |


      v


collect_papers() -> 10 parsers -> list of articles


      |


      v


analyze_papers() -> 3 checks -> DataFrame with verdicts


      |


      v


export_csv() / export_excel() -> file download


```


**Links:**


- Run in Colab: https://colab.research.google.com/github/DmitPerson42/science-paper-analyzer/blob/main/science_paper_analyzer.ipynb


- This tutorial: https://colab.research.google.com/github/DmitPerson42/science-paper-analyzer/blob/main/tutorial.ipynb


- GitHub: https://github.com/DmitPerson42/science-paper-analyzer
